In [ ]:
import pandas as pd
from typing import Any
from pygbif import occurance as occ
from pygbif import registry

def _record_columns() -> tuple[str, ...]:
    """Return the stable schema used for image-level GBIF records."""
    return (
        "source_group_id",
        "scientific_name",
        "accepted_scientific_name",
        "taxon_key",
        "species",
        "species_key",
        "genus",
        "family",
        "order",
        "class",
        "phylum",
        "kingdom",
        "year",
        "event_date",
        "basis_of_record",
        "country_code",
        "decimal_latitude",
        "decimal_longitude",
        "occurrence_source_url",
        "image_url",
        "image_license",
        "image_creator",
        "image_references",
        "image_type",
        "image_format",
        "image_created",
        "dataset_key",
        "publishing_org_key",
        "dataset_citation",
        "dataset_doi",
    )

def _get_all_records(**params) -> list[dict[str, Any]]:
    """Fetch occurrences and project their media into a fixed image schema.

    Each returned dictionary represents one image. Occurrence metadata is
    repeated for all images belonging to the same occurrence, whose GBIF
    key is exposed as ``source_group_id`` for group-aware dataset splits.
    Missing GBIF fields are represented by ``None``.

    Returns:
        Image-level records with stable occurrence, image, taxonomy, and
        source-dataset fields.
    """
    image_records: list[dict[str, Any]] = []
    dataset_metadata: dict[str, tuple[Any, Any]] = {}
    offset = 0
    page_size = 300

    while True:
        response = occ.search(
            **params,
            limit=page_size,
            offset=offset,
        )
        page = response["results"]

        for occurrence in page:
            dataset_key = occurrence.get("datasetKey")
            if dataset_key and dataset_key not in dataset_metadata:
                dataset = registry.datasets(uuid=dataset_key)
                citation_value = dataset.get("citation")
                citation = (
                    citation_value.get("text")
                    if isinstance(citation_value, dict)
                    else citation_value
                )
                dataset_metadata[dataset_key] = (citation, dataset.get("doi"))

            citation, doi = dataset_metadata.get(dataset_key, (None, None))
            for media in occurrence.get("media", []):
                if media.get("type") != media_type:
                    continue

                image_url = media.get("identifier")
                if not image_url:
                    continue

                image_records.append(
                    {
                        "source_group_id": occurrence.get("key"),
                        "scientific_name": occurrence.get("scientificName"),
                        "accepted_scientific_name": occurrence.get(
                            "acceptedScientificName"
                        ),
                        "taxon_key": occurrence.get("taxonKey"),
                        "species": occurrence.get("species"),
                        "species_key": occurrence.get("speciesKey"),
                        "genus": occurrence.get("genus"),
                        "family": occurrence.get("family"),
                        "order": occurrence.get("order"),
                        "class": occurrence.get("class"),
                        "phylum": occurrence.get("phylum"),
                        "kingdom": occurrence.get("kingdom"),
                        "year": occurrence.get("year"),
                        "event_date": occurrence.get("eventDate"),
                        "basis_of_record": occurrence.get("basisOfRecord"),
                        "country_code": occurrence.get("countryCode"),
                        "decimal_latitude": occurrence.get("decimalLatitude"),
                        "decimal_longitude": occurrence.get("decimalLongitude"),
                        "occurrence_source_url": occurrence.get("references"),
                        "image_url": image_url,
                        "image_license": media.get("license"),
                        "image_creator": media.get("creator"),
                        "image_references": media.get("references"),
                        "image_type": media.get("type"),
                        "image_format": media.get("format"),
                        "image_created": media.get("created"),
                        "dataset_key": dataset_key,
                        "publishing_org_key": occurrence.get("publishingOrgKey"),
                        "dataset_citation": citation,
                        "dataset_doi": doi,
                    }
                )

        if response["endOfRecords"] or not page:
            break

        offset += len(page)

    return image_records

def _get_species_records(species: str) -> pd.DataFrame:
    """_summary_

    Args:
        species (str): _description_

    Returns:
        pd.DataFrame: _description_
    """
    records: list[dict[str, Any]] = []
    for year in years:
        logger.info("Fetching %s observations from %s", species, year)
        records_for_year = _get_all_records(
            scientificName=species,
            mediatype=media_type,
            year=year,
        )
        records.extend(records_for_year)

    records_df = pd.DataFrame.from_records(records, columns=_record_columns())
    if records_df.empty:
        return records_df
    # Random sampling prevents the API's ordering from biasing a capped dataset.
    records_df = records_df.sample(frac=1, random_state=seed).reset_index(
        drop=True
    )

    # cap data if bigger than limit
    if len(records_df) > max_img_num:
        records_df = records_df.head(max_img_num)

    return records_df